# 16 · SkewerJAX and fake_spectra: memory, scaling and stage by stage

Notebook 14 follows the time of every stage of the four entries. This notebook follows their **memory**: how much each
code and mode uses, **where** (host RAM against GPU VRAM), **in which stage**, and how it scales. The GPU entry still runs
the pass, the crossings and the contributions on the host, so it uses both host and device memory, and the figures keep
the two apart. The blocks are those of notebook 14, and one more:

- **A, the simulation:** TNG50-4, TNG50-3 and TNG50-2 at $128^2$ skewers, 1 km/s;
- **B, the number of skewers:** TNG50-2, 16 to 65,536 skewers;
- **C1, the pixel width:** TNG50-2, $128^2$ skewers, 1 to 16 km/s (fake_spectra, default CPU and GPU);
- **D, SkewerJAX's memory knobs** at $128^2$ TNG50-2 skewers: the tile size (256, 1,024, 4,096), one or two physics
  configurations in the pass, the threads of the pass (8, 16, 32); their time cost from the same runs.

**By default the notebook reads the records** of `scripts/benchmarks/benchmark_memory.py` (`benchmarks/results/memory/`,
one JSON per (point, entry): the sampled timeline, the stage markers, the peaks, the pass's and tiles' facts) and runs in
under a minute. **`RUN = True` re-runs the benchmark** for the (point, entry) pairs without a record (resumable; each in
its own fresh process): about 3 hours on this machine for all of them. Notebook 15, §6, uses the same records for the
CPU utilisation and the bytes read of the two codes.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")             # this kernel reads the records and plots; the benchmark workers set their own
import inspect, logging
logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (SCALING_ENTRY_STYLES, PALETTE, add_panel_letters, plot_memory_knobs, plot_memory_peaks,
                                      plot_memory_scaling, plot_memory_timelines, use_backend, use_style)
from skewerjax.utils.paths import BENCHMARK_RECORDS_DIR, DATA_DIR
from skewerjax.comparison import memory as M, scaling as S
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

## 1 · The protocol

**Host memory has two kinds**, and the figures always keep them apart:

- **anonymous** memory (`RssAnon` of `/proc/<pid>/status`): what the code allocates, its arrays and the interpreter's;
  it is what can exhaust the RAM;
- **file-backed** pages (`RssFile`): the pages of files mapped into the process (the memory-mapped reads of the pass,
  the shared libraries); the kernel can drop them at any time, so they do not count against the RAM in the same way.

`VmHWM` is the process's peak resident set (both kinds) since it started; the system's `MemAvailable` is sampled too.

**GPU memory** is the process's total from NVML, which includes the CUDA context, and JAX's `memory_stats()` at every
stage marker (`bytes_in_use`, `peak_bytes_in_use`; `XLA_PYTHON_CLIENT_PREALLOCATE=false`, so the pool grows on demand).
The launcher runs JAX on the CPU and never holds the GPU; a CPU entry's worker has no CUDA context at all.

**One fresh process per (point, entry)**, cold (the snapshot dropped from the page cache before the measured run), with
notebook 14's configuration: $\mathrm{Ly}\alpha$ and Si III 1206, float32, 32 threads, the CPU as two JAX devices. A SkewerJAX entry first
compiles its kernels on TNG50-4 (the warm-up of notebook 14, marked "warm-up"); its peaks are those of the measured run,
from the "setup" marker on. It runs its own physics configuration only (notebook 14 shared one pass with both). One
sample of the sampler:

In [ ]:
print(inspect.getsource(M.ProcessSampler._sample))

## The records

`RUN = False` (the default) reads the records; `True` measures every (point, entry) without one, then reads them all (the
same as `python scripts/benchmarks/benchmark_memory.py`). `RUN_ONLY` restricts a run to some ids (e.g. one small point).

In [ ]:
RUN = False                                   # True: run the benchmark for the (point, entry) pairs without a record (resumes)
RUN_ONLY = None                               # e.g. ["TNG50-2_n4_dv1_cpu"]: only these
OUT = BENCHMARK_RECORDS_DIR / "memory"        # one JSON per (point, entry) ($SKEWERJAX_BENCHMARK_RESULTS)
POINTS = M.memory_points()
if RUN:
    M.run_points([c for c in POINTS if RUN_ONLY is None or c["id"] in RUN_ONLY], OUT, data_dir=DATA_DIR)
R = M.load_memory_records(OUT)
print(f"{len(R)} of {len(POINTS)} (point, entry) pairs recorded in {OUT}")
wall = 0.0
for c in POINTS:
    r = R.get(c["id"])
    if r is None:
        print(f"  {c['id']:34s} ({', '.join(c['blocks'])}): not recorded")
        continue
    wall += r["wall_s"]
    pk = r["peak"]
    gpu = f", GPU {pk['gpu_GB']:.2f} GB" if pk.get("gpu_GB") else ""
    print(f"  {c['id']:34s} ({', '.join(c['blocks']):8s}) {r['date']}, cache {r['cache']:.0%}: anonymous {pk['anon_GB']:6.2f} GB, "
          f"file-backed {pk['file_GB']:5.2f} GB{gpu}; read {r['read_GB']:6.1f} GB; {r['wall_s']:5.0f} s")
print(f"measured runs: {wall / 3600:.1f} h in all (the warm-ups and process starts besides)")
ID = lambda sim, n, dv, e, **kw: M.point_id(sim, n, dv, e, **kw)
ENTRIES = (S.FS, S.CPU_DEF, S.GPU_DEF, S.CPU_MAT)

## 2 · Memory against time

The four entries at $128^2$ TNG50-2 skewers, from the start of the measured run: host anonymous (solid) and file-backed
(dashed) memory on a log axis, the GPU's (green, right axis) for the GPU entry, the stages shaded, this machine's limits
dash-dotted (125 GB of RAM, 12 GB of VRAM).

In [ ]:
recs = {e: R[ID("TNG50-2", 128, 1.0, e)] for e in ENTRIES if ID("TNG50-2", 128, 1.0, e) in R}
fig, axs = plot_memory_timelines(recs, ram_GB=M.MACHINE_RAM_GB, vram_GB=M.MACHINE_VRAM_GB, figsize=(22, 4.4 * len(recs) + 1.5))
fig.suptitle("fake_spectra holds one file at a time; SkewerJAX holds the cells within reach, and three tiles in preparation", fontsize=21)
plt.show()
for e, r in recs.items():
    print(f"{e:24s} " + "; ".join(f"{st}: {p['anon_GB']:.1f} GB anon, {p['file_GB']:.2f} GB file"
                                  + (f", GPU {p['gpu_GB']:.2f} GB" if p.get('gpu_GB') and np.isfinite(p['gpu_GB']) else "") + f" ({p['duration_s']:.0f} s)"
                                  for st, p in r["peaks"].items() if st not in ("start", "warm-up", "done")))

*Reading.* fake_spectra's memory saws up and down file by file: it reads one file's datasets, converts them, hands them to
C and drops them, for each line; its peak, 2.8 GB, is one file's arrays and its float64 spectra. SkewerJAX's pass climbs
to 27 GB while its blocks collect the cells within reach; the assembled store (19.8 GB) stays through the tiles, which add
$\sim$8.5 GB: three tiles in preparation on the host, each with its candidate pairs and a float64 copy of its cells. The
file-backed pages stay near a gigabyte: the memory-mapped gathers touch pages briefly, and the kernel drops them. The GPU
entry's host memory follows the CPU entry's (the same pass and the same host work); its device memory, 1.6 GB, is the CUDA
context and the pool XLA grew for one tile's buffers during the warm-up, which it keeps for the run.

## 3 · Peaks by component

At four numbers of skewers, every entry's peak anonymous memory split into what the code knows
(`comparison.memory.peak_components`: the baseline when the run starts, the store and the pass's blocks or the tiles'
contributions and spectra, fake_spectra's per-file and $\tau$ arrays; the rest unaccounted), and the GPU's apart.

In [ ]:
bars = []
for n in (4, 32, 128, 256):
    for e in ENTRIES:
        r = R.get(ID("TNG50-2", n, 1.0, e))
        if r is None:
            continue
        pc = M.peak_components(r)
        bars.append(dict(label=f"{n * n:,} sk, {e.replace('SkewerJAX ', '')} ({pc['stage']})", host=pc["host"], gpu=pc["gpu"]))
fig, axs = plot_memory_peaks(bars, figsize=(22, 0.55 * len(bars) + 4))
fig.suptitle("Below ~4,096 skewers the pass's read buffers set SkewerJAX's peak; above, the store and the tiles in preparation", fontsize=21)
plt.show()
for b in bars:
    print(f"{b['label']:44s} " + ", ".join(f"{k} {v:.2f}" for k, v in b["host"].items()) + (f"; GPU " + ", ".join(f"{k} {v:.2f}" for k, v in b["gpu"].items()) if b["gpu"] else ""))

*Reading.* The bar of each entry is its peak, in the stage written after its name. Below $\sim$4,096 skewers SkewerJAX's
peak is in the pass: 32 threads, each holding a block of 2 million rows (the geometry, its float64 conversion and the rest
of the needed rows), $\sim$5 GB whatever the number of skewers; fake_spectra's is one file's arrays, $\sim$1 GB, and its
spectra. At 16,384 skewers SkewerJAX's peak is in the tiles: the store (19.8 GB) and three tiles in preparation
($\sim$8.7 GB: an estimate from the code, scaled to the measurement); at 65,536 it is the store's assembly (50 GB of store,
the blocks' chunks not yet freed). "Unaccounted" is what the estimates leave: the allocator's retention, Python's objects,
temporaries. The GPU holds its CUDA context (0.1 GB) and one tile's buffers.

## 4 · Scaling per block

The peak of every entry against each block's parameter, with fitted models: fake_spectra's anonymous peak as a fixed part
plus bytes per skewer (its spectra) or per cell of a file; SkewerJAX's as a fixed part plus bytes per cell within reach
(the store), the cells within reach following a coverage law in the number of skewers,
$n_{\rm reach}(N) = N_{\rm cells}\,[1 - e^{-kN}]$, fitted to the measured counts. The fits are extrapolated to $512^2$
skewers (block B), where this machine's RAM would be reached.

In [ ]:
def series(ids):
    out = {}
    for e in ENTRIES:
        rs = [R.get(i(e)) for i in ids]
        out[e] = dict(anon=[r["peak"]["anon_GB"] if r else np.nan for r in rs], file=[r["peak"]["file_GB"] if r else np.nan for r in rs],
                      gpu=[(r["peak"]["gpu_GB"] or np.nan) if r else np.nan for r in rs])
    return out

nB = np.array(M.B_SIDES)
NB = nB**2
idsB = [lambda e, n=n: ID("TNG50-2", n, 1.0, e) for n in nB]
serB = series(idsB)
n_gas = next(r["snapshot"]["n_gas"] for r in R.values() if r["config"]["sim"] == "TNG50-2")
need = {}
for e in S.SKEWERJAX_ENTRIES:
    rs = [R.get(i(e)) for i in idsB]
    need[e] = np.array([r["pass_info"]["n_needed"] if r else np.nan for r in rs], float)
ok = np.isfinite(need[S.CPU_DEF])
k_cov = float(np.median(-np.log(1 - need[S.CPU_DEF][ok] / n_gas) / NB[ok]))   # the coverage law's k
reach = lambda N: n_gas * (1 - np.exp(-k_cov * np.asarray(N, float)))
fitsB = {S.FS: {"anon": M.fit_linear(NB, serB[S.FS]["anon"])}}
for e in S.SKEWERJAX_ENTRIES:
    a, b = M.fit_linear(need[e], serB[e]["anon"])
    fitsB[e] = {"anon": (lambda N, a=a, b=b: a + b * reach(N))}
    print(f"{e:24s} anonymous peak = {a:5.2f} GB + {1e9 * b:5.1f} bytes x cells within reach")
a, b = fitsB[S.FS]["anon"]
print(f"{S.FS:24s} anonymous peak = {a:5.2f} GB + {1e9 * b:,.0f} bytes x skewers (its float64 spectra: {3 * 2 * 3961 * 8:,} bytes per skewer)")
print(f"cells within reach: k = {k_cov:.2e}; at 512² skewers {reach(512**2) / n_gas:.0%} of the {n_gas:,} cells")
fig, axs = plot_memory_scaling(NB, serB, xlabel="skewers (TNG50-2)", fits=fitsB, limits=(M.MACHINE_RAM_GB, M.MACHINE_VRAM_GB),
                               extrapolate=512**2, figsize=(22, 7.5))
fig.suptitle("Block B: SkewerJAX's store grows with the cells within reach; at 512² it would hold nearly every cell", fontsize=21)
plt.show()
for e in ENTRIES:
    f = fitsB[e]["anon"]
    y = f(512**2) if callable(f) else f[0] + f[1] * 512**2
    print(f"{e:24s} extrapolated to 512²: {y:5.0f} GB anonymous ({'above' if y > M.MACHINE_RAM_GB else 'below'} the {M.MACHINE_RAM_GB:.0f} GB of RAM)")

*Reading.* (a) fake_spectra's anonymous peak grows slowly, with its float64 spectra (three of $N \times N_v$ at once: the
cached H I, the result and the current file's part); SkewerJAX's is flat at the pass's read buffers up to $\sim$4,096
skewers, then follows the store. The store's slope, in bytes per cell within reach, is printed above; the coverage law
puts 98% of the TNG50-2 cells within reach at $512^2$: the extrapolation, 113–116 GB, falls just below this machine's
125 GB (dash-dotted), with no margin for the system or a second process.
(b) The file-backed pages stay below a gigabyte for both codes. (c) The GPU holds one tile at a time: its memory does not
grow with the number of skewers beyond the first tile.

In [ ]:
sims = ("TNG50-4", "TNG50-3", "TNG50-2")
idsA = [lambda e, s=s: ID(s, 128, 1.0, e) for s in sims]
serA = series(idsA)
xA = np.array([next((r["snapshot"]["n_gas"] for r in R.values() if r["config"]["sim"] == s), np.nan) for s in sims], float)
fig, axs = plot_memory_scaling(xA, serA, xlabel="gas cells in the snapshot (128² skewers)", fits=None, limits=(M.MACHINE_RAM_GB, M.MACHINE_VRAM_GB),
                               figsize=(22, 7.5))
fig.suptitle("Block A: fake_spectra's peak follows the cells of one file, SkewerJAX's the cells within reach", fontsize=21)
plt.show()
for s, x in zip(sims, xA):
    r = R.get(ID(s, 128, 1.0, S.CPU_DEF))
    if r:
        print(f"{s}: {x:,.0f} cells, {r['pass_info']['n_needed']:,} within reach ({r['pass_info']['fraction']:.0%}); store {sum(r['store_bytes'].values()) / 1e9:.1f} GB "
              f"({sum(r['store_bytes'].values()) / r['pass_info']['n_needed']:.0f} bytes per cell)")

*Reading.* The three simulations fill the same box with 8 times more cells each. fake_spectra's peak follows the size of
one file (it holds one at a time); SkewerJAX's follows the cells within reach of $128^2$ skewers: 86% of TNG50-4's large
cells, 23% of TNG50-2's small ones, at the bytes per cell printed above.

In [ ]:
dvs = (1.0, 2.0, 4.0, 8.0, 16.0)
idsC = [lambda e, d=d: ID("TNG50-2", 128, d, e) for d in dvs]
serC = series(idsC)
fig, axs = plot_memory_scaling(np.array(dvs), serC, xlabel="pixel width [km/s] (128² TNG50-2 skewers)", fits=None,
                               limits=(M.MACHINE_RAM_GB, M.MACHINE_VRAM_GB), figsize=(22, 7.5))
fig.suptitle("Block C1: the pixels set only the spectra, a small part of either code's peak", fontsize=21)
plt.show()
for d in dvs:
    print(f"{d:4g} km/s: " + "; ".join(f"{e} {R[ID('TNG50-2', 128, d, e)]['peak']['anon_GB']:.1f} GB"
                                      + (f" (GPU {R[ID('TNG50-2', 128, d, e)]['peak']['gpu_GB']:.2f})" if e == S.GPU_DEF else "")
                                      for e in (S.FS, S.CPU_DEF, S.GPU_DEF) if ID("TNG50-2", 128, d, e) in R))

*Reading.* Wider pixels mean fewer of them: the spectra shrink as $1/dv$ ($N \times N_v \times$ 2 lines $\times$ 4 bytes for
SkewerJAX, 8 bytes and three arrays for fake_spectra), and the windows of the engine too; the store, the larger part,
does not depend on the pixels.

## 5 · The knobs

SkewerJAX's memory knobs at $128^2$ TNG50-2 skewers, each run's peak against its wall time (the measured run, after the
warm-up): the tile size (`tile_size`: the skewers of a tile, and so the contributions and the GPU buffers in flight), two
physics configurations in one pass instead of one (16 bytes per cell more), and the threads of the pass (the blocks in
flight).

In [ ]:
knobs = [("tile 256", ID("TNG50-2", 128, 1.0, S.CPU_DEF, tile_size=256)), ("tile 1,024 (default)", ID("TNG50-2", 128, 1.0, S.CPU_DEF)),
         ("tile 4,096", ID("TNG50-2", 128, 1.0, S.CPU_DEF, tile_size=4096)), ("two physics configurations", ID("TNG50-2", 128, 1.0, S.CPU_DEF, physics="both")),
         ("8 pass threads", ID("TNG50-2", 128, 1.0, S.CPU_DEF, threads=8)), ("16 pass threads", ID("TNG50-2", 128, 1.0, S.CPU_DEF, threads=16)),
         ("GPU, tile 256", ID("TNG50-2", 128, 1.0, S.GPU_DEF, tile_size=256)), ("GPU, tile 1,024", ID("TNG50-2", 128, 1.0, S.GPU_DEF)),
         ("GPU, tile 4,096", ID("TNG50-2", 128, 1.0, S.GPU_DEF, tile_size=4096))]
OFFSETS = {"8 pass threads": (-8, 8), "16 pass threads": (-6, -22), "tile 1,024 (default)": (8, 6), "tile 256": (8, -6)}   # the labels' places
rows = []
for lab, i in knobs:
    r = R.get(i)
    if r is None:
        continue
    rows.append(dict(label=lab, anon_GB=r["peak"]["anon_GB"], gpu_GB=r["peak"].get("gpu_GB"), time_s=r["run_s"],
                     color=PALETTE["green"] if "GPU" in lab else PALETTE["blue"], offset=OFFSETS.get(lab, (8, 4))))
fig, axs = plot_memory_knobs(rows, reference="tile 1,024 (default)", figsize=(22, 7.5))
fig.suptitle("Smaller tiles save 3 GB for 4% of the time, larger ones cost both; the pass threads change little", fontsize=21)
plt.show()
ref = next(r for r in rows if r["label"] == "tile 1,024 (default)")
for r in rows:
    print(f"{r['label']:28s} anonymous {r['anon_GB']:6.2f} GB ({r['anon_GB'] - ref['anon_GB']:+6.2f}), time {r['time_s']:6.1f} s "
          f"({r['time_s'] - ref['time_s']:+6.1f})" + (f", GPU {r['gpu_GB']:.2f} GB" if r.get("gpu_GB") else ""))

*Reading.* Against the default (tiles of 1,024 skewers, one physics configuration, 32 pass threads; 29.8 GB, 152 s):

- **the tile size** sets the tiles in preparation: tiles of 256 skewers save 3.1 GB (the peak moves to the store's
  assembly) for 6.6 s more; tiles of 4,096 cost 27 GB more and 19 s (more pairs and cells per tile). On the GPU the same,
  and the device's peak follows the tile: 1.2, 1.6 and 5.8 GB;
- **two physics configurations** in one pass cost 4.3 GB (16 bytes per cell within reach), and no time;
- **fewer pass threads** save little memory here (the peak is in the tiles) and no time: 8 threads (142 s) read as fast
  as 32 from this disk, within the run-to-run noise of the reads.

## 6 · Discussion

The peaks at four numbers of skewers, host anonymous / file-backed and GPU, per entry:

In [ ]:
print(f"{'skewers':>8s}  " + "  ".join(f"{e:>30s}" for e in ENTRIES))
for n in (4, 32, 128, 256):
    cells = []
    for e in ENTRIES:
        r = R.get(ID("TNG50-2", n, 1.0, e))
        cells.append("-" if r is None else f"{r['peak']['anon_GB']:.1f} / {r['peak']['file_GB']:.2f}" + (f" / GPU {r['peak']['gpu_GB']:.2f}" if r['peak'].get('gpu_GB') else "") + " GB")
    print(f"{n * n:8,d}  " + "  ".join(f"{c:>30s}" for c in cells))
r = R.get(ID("TNG50-2", 128, 1.0, S.CPU_DEF))
if r:
    sb, n_need = r["store_bytes"], r["pass_info"]["n_needed"]
    per = {k: v / n_need for k, v in sb.items()}
    print(f"\nthe store at 128²: {n_need:,} cells, " + ", ".join(f"{k} {v:.1f} B" for k, v in per.items()) + f" per cell ({sum(per.values()):.0f} B)")
    big = R.get(ID("TNG50-2", 256, 1.0, S.CPU_DEF))
    nb = big["pass_info"]["n_needed"] if big else n_need
    pc = M.peak_components(r)
    print(f"the tiles in preparation at 128²: {pc['host']['tiles']:.1f} GB (three tiles: candidate pairs and float64 cells)")
    print("ways to lower the peak, estimated at 256² (" + f"{nb:,} cells within reach):")
    for what, b in (("one physics configuration per pass (the benchmark of notebook 14 ran two)", 16),
                    ("float32 positions relative to the tile (instead of float64)", 12), ("no ids in the store (only the cells' order is needed)", 8),
                    ("(cell, tile) pairs as int32", 4 * per["tile_pairs"] / 8)):
        print(f"  {what:72s} {b:5.1f} B per cell: {b * nb / 1e9:5.1f} GB")
    th = r["pass_info"]["threads"]
    print(f"  the pass's blocks in flight: {th} threads x 2 million rows x ~100 B = {th * 2e6 * 100 / 1e9:.1f} GB (halved with half the rows per block)")

**Which stage sets each peak, and why.**

- **SkewerJAX, default CPU and matched**: below $\sim$4,096 TNG50-2 skewers, the pass: 32 threads each hold a block of
  2 million rows (the geometry of every row, its float64 conversion, the other datasets of the needed rows), $\sim$5 GB
  whatever $N$. At $128^2$, the tiles: the store (72 bytes per cell within reach) and three tiles in preparation (each
  with its $\sim$17,000 candidate pairs per skewer and a float64 copy of its cells). At $256^2$, the store's assembly
  (the blocks' chunks of needed cells, copied into the store and freed one by one).
- **SkewerJAX, GPU**: the same host memory as the CPU entry (the pass, the crossings and the contributions run on the
  host); on the device, the CUDA context and one tile's buffers (its contributions, its $\tau$, the far-wing transform),
  set by the tile size, not by $N$.
- **fake_spectra** holds one file at a time (`spectra.py:_interpolate_single_file`: the datasets of one file and their
  float64 copies for C, $\sim$96 bytes per cell, $\sim$0.9 GB for TNG50-2's 9.4 million cells per file) and its spectra in
  float64 (the result, the current file's part and the cached H I): its peak grows with $N$ only through the spectra
  (7.5 GB at 65,536 skewers). It pays for this low memory with time: it reads and converts every cell, for each line
  (notebook 15, §6).

**Ranked ways to lower SkewerJAX's memory** (the estimates above; at $128^2$ and $256^2$ TNG50-2 skewers):

1. **The tiles in preparation** ($\sim$8.7 GB at $128^2$): each copies its cells to float64 `Cells` and keeps its
   candidate pairs with a sorted copy. Float32 views of the store, the pairs sorted in place, or one tile prepared ahead
   instead of three would save most of it; smaller tiles already save 3 GB for 4% of the time (§5).
2. **One physics configuration per pass** (16 bytes per cell: 4.4 GB at $128^2$, 11 GB at $256^2$): what this benchmark
   runs; notebook 14's shared pass held both.
3. **Float32 positions relative to the tile** (12 bytes per cell: 3.3 / 8.3 GB): the crossings need $\sim$1e-7 of a cell
   size, which float32 offsets within a tile give.
4. **No ids, and int32 (cell, tile) pairs** (12 bytes per cell: 3.3 / 8.3 GB).
5. **Fewer rows per block** at small $N$ (the $\sim$5 GB of blocks in flight).

Items 3 and 4 cut the store from 72 to 48 bytes per cell: at $512^2$ skewers (98% of TNG50-2's cells within reach) the
store would need $\sim$57 GB instead of $\sim$85 GB, and with item 1 the peak would fall from the extrapolated
$\sim$115 GB to $\sim$75 GB, a comfortable margin on this machine.